## Load the required packages

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import anndata  
import os
from pathlib import Path
from scipy.sparse import csr_matrix
import scipy.sparse as sp

Global seed set to 0


## Set path

In [2]:
working_dir = Path(os.getcwd())
dataset_dir = f"{working_dir.parents[1]}/datasets/CITE-SLN-Gayoso"
base_dir = f"{working_dir.parents[1]}/outputs/batch variation/208_M2-208_M1/TotalVI"
os.makedirs(base_dir, exist_ok=True)

## Load data

In [3]:
train_data = anndata.read(f"{dataset_dir}/SLN208_M2.h5ad")   
test_data = anndata.read(f"{dataset_dir}/SLN208_M1.h5ad")   
train_data, test_data

(AnnData object with n_obs × n_vars = 7105 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 AnnData object with n_obs × n_vars = 8715 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

## Select common genes

In [4]:
common_genes = np.intersect1d(train_data.var.index, test_data.var.index)
train_data = train_data[:, common_genes]
test_data = test_data[:, common_genes]

## Select common proteins

In [5]:
common_proteins = np.intersect1d(np.array(train_data.uns["protein_names"]), np.array(test_data.uns["protein_names"]))
if isinstance(train_data.obsm["protein_expression"], pd.DataFrame):
    train_protein_expression = train_data.obsm["protein_expression"]
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = train_protein_expression
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)
else:
    train_protein_expression = pd.DataFrame(train_data.obsm["protein_expression"].todense(), index=train_data.obs.index, columns=train_data.uns["protein_names"])
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = csr_matrix(train_protein_expression.values)
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)

if isinstance(test_data.obsm["protein_expression"], pd.DataFrame):
    test_protein_expression = test_data.obsm["protein_expression"]
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = test_protein_expression
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)
else:
    test_protein_expression = pd.DataFrame(test_data.obsm["protein_expression"].todense(), index=test_data.obs.index, columns=test_data.uns["protein_names"])
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = csr_matrix(test_protein_expression.values)
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)

In [6]:
if sp.issparse(train_data.X):
    train_rna_expression = pd.DataFrame(train_data.X.todense(), columns=train_data.var.index, index=train_data.obs.index)
else:
    train_rna_expression = pd.DataFrame(train_data.X, columns=train_data.var.index, index=train_data.obs.index)

if sp.issparse(test_data.X):
    test_rna_expression = pd.DataFrame(test_data.X.todense(), columns=test_data.var.index, index=test_data.obs.index)
else:
    test_rna_expression = pd.DataFrame(test_data.X, columns=test_data.var.index, index=test_data.obs.index)
train_rna_expression, test_rna_expression

(index               0610007P14Rik  0610009B22Rik  0610009L18Rik  \
 index                                                             
 AAACCCAAGGTATCTC-2            0.0            0.0            0.0   
 AAACCCACATTCTGTT-2            0.0            0.0            0.0   
 AAACCCAGTAGCTTGT-2            0.0            0.0            0.0   
 AAACCCAGTATCAGCT-2            0.0            0.0            0.0   
 AAACCCAGTTTCTATC-2            0.0            0.0            0.0   
 ...                           ...            ...            ...   
 TTTGTTGGTGGGCTCT-2            0.0            0.0            0.0   
 TTTGTTGTCAAATAGG-2            0.0            0.0            0.0   
 TTTGTTGTCACGAGGA-2            0.0            0.0            0.0   
 TTTGTTGTCTCGACCT-2            0.0            0.0            0.0   
 TTTGTTGTCTTGGTCC-2            0.0            0.0            0.0   
 
 index               0610009O20Rik  0610010F05Rik  0610010K14Rik  \
 index                                        

In [7]:
rna_expression = pd.concat([train_rna_expression, test_rna_expression], axis=0)
rna_expression

index,0610007P14Rik,0610009B22Rik,0610009L18Rik,0610009O20Rik,0610010F05Rik,0610010K14Rik,0610011F06Rik,0610012G03Rik,0610030E20Rik,0610037L13Rik,...,mt-Co2,mt-Co3,mt-Cytb,mt-Nd1,mt-Nd2,mt-Nd3,mt-Nd4,mt-Nd4l,mt-Nd5,mt-Nd6
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGGTATCTC-2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,17.0,14.0,6.0,7.0,11.0,9.0,6.0,4.0,3.0,0.0
AAACCCACATTCTGTT-2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,15.0,15.0,2.0,4.0,4.0,8.0,3.0,1.0,0.0,0.0
AAACCCAGTAGCTTGT-2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,18.0,17.0,8.0,5.0,4.0,9.0,5.0,1.0,2.0,0.0
AAACCCAGTATCAGCT-2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,16.0,18.0,4.0,4.0,4.0,11.0,3.0,0.0,0.0,0.0
AAACCCAGTTTCTATC-2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14.0,31.0,6.0,3.0,5.0,12.0,1.0,2.0,2.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,37.0,31.0,15.0,7.0,9.0,2.0,5.0,3.0,0.0,0.0
TTTGTTGGTGCTCTTC-1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,51.0,68.0,25.0,12.0,11.0,5.0,13.0,1.0,4.0,0.0
TTTGTTGTCAGCCTTC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,43.0,38.0,8.0,9.0,12.0,2.0,6.0,0.0,3.0,0.0


In [8]:
protein_expression = pd.concat([train_protein_expression, test_protein_expression], axis=0)   
protein_expression

,ADT_4-1BBLigand(CD137L)_A0890,ADT_CD102_A0104,ADT_CD103_A0201,ADT_CD105_A0812,ADT_CD106_A0226,ADT_CD107a(LAMP-1)_A0905,ADT_CD115(CSF-1R)_A0105,ADT_CD117(c-Kit)_A0012,ADT_CD11a_A0595,ADT_CD11b_A0014,...,ADT_TCRVr3_A0210,ADT_TCRbchain_A0120,ADT_TCRr-d_A0121,ADT_TER-119-ErythroidCells_A0122,ADT_TIGIT(Vstm3)_A0848,ADT_TLR4(CD284)-MD2Complex_A0875,ADT_Tim-4_A0567,ADT_XCR1_A0568,ADT_anti-P2RY12_A0415,ADT_integrinb7_A0214
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGGTATCTC-2,1.0,11.0,8.0,4.0,1.0,1.0,0.0,6.0,70.0,1.0,...,6.0,59.0,1.0,2.0,15.0,3.0,12.0,4.0,5.0,15.0
AAACCCACATTCTGTT-2,3.0,17.0,1.0,3.0,5.0,12.0,5.0,3.0,40.0,3.0,...,6.0,0.0,2.0,0.0,10.0,2.0,11.0,3.0,4.0,31.0
AAACCCAGTAGCTTGT-2,5.0,19.0,3.0,4.0,1.0,3.0,4.0,6.0,66.0,1.0,...,5.0,52.0,3.0,1.0,6.0,0.0,4.0,1.0,4.0,21.0
AAACCCAGTATCAGCT-2,1.0,33.0,6.0,4.0,3.0,0.0,4.0,5.0,36.0,2.0,...,10.0,4.0,0.0,1.0,8.0,5.0,5.0,5.0,2.0,32.0
AAACCCAGTTTCTATC-2,0.0,18.0,5.0,5.0,2.0,7.0,1.0,2.0,32.0,6.0,...,7.0,1.0,5.0,0.0,14.0,1.0,1.0,3.0,8.0,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,3.0,22.0,7.0,3.0,0.0,0.0,7.0,3.0,20.0,4.0,...,2.0,1.0,5.0,1.0,8.0,4.0,3.0,3.0,2.0,7.0
TTTGTTGGTGCTCTTC-1,1.0,17.0,8.0,6.0,8.0,2.0,8.0,10.0,37.0,4.0,...,11.0,6.0,8.0,1.0,26.0,7.0,11.0,8.0,4.0,15.0
TTTGTTGTCAGCCTTC-1,0.0,23.0,5.0,3.0,1.0,3.0,2.0,5.0,56.0,3.0,...,10.0,27.0,3.0,1.0,7.0,5.0,4.0,0.0,3.0,20.0


In [9]:
cells = pd.concat([train_data.obs, test_data.obs], axis=0)
cells["batch_index"] = np.repeat(0, cells.shape[0])   
cells["batch_index"][train_data.shape[0]:] = 1
cells

/tmp/ipykernel_40873/2780980773.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cells["batch_index"][train_data.shape[0]:] = 1


,n_protein_counts,n_proteins,seurat_hash_id,batch_indices,hash_id,n_genes,percent_mito,leiden_subclusters,cell_types,batch_index
index,,,,,,,,,,
AAACCCAAGGTATCTC-2,2871.0,192,Spleen,1,Spleen,1276,0.036945,0,CD4 T,0
AAACCCACATTCTGTT-2,2615.0,191,Lymph_Node,1,Lymph Node,758,0.042187,1,Mature B,0
AAACCCAGTAGCTTGT-2,2503.0,189,Lymph_Node,1,Lymph Node,926,0.050122,14,Ifit3-high CD4 T,0
AAACCCAGTATCAGCT-2,2239.0,194,Lymph_Node,1,Lymph Node,668,0.058909,5,Mature B,0
AAACCCAGTTTCTATC-2,2185.0,186,Lymph_Node,1,Lymph Node,839,0.062933,1,Mature B,0
...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,2455.0,186,Spleen,0,Spleen,1334,0.052716,1,Mature B,1
TTTGTTGGTGCTCTTC-1,3702.0,203,Lymph_Node,0,Lymph Node,1388,0.076520,21,B doublets,1
TTTGTTGTCAGCCTTC-1,2881.0,193,Lymph_Node,0,Lymph Node,1874,0.039662,0,CD4 T,1


In [10]:
data = anndata.AnnData(X=rna_expression, obs=cells)
data.obsm["protein_expression"] = protein_expression
data.uns["protein_names"] = train_data.uns["protein_names"]
data.write(f"{base_dir}/TotalVI_data.h5ad", compression="gzip") 
data

AnnData object with n_obs × n_vars = 15820 × 13553
    obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types', 'batch_index'
    uns: 'protein_names'
    obsm: 'protein_expression'

## Mask the protein expression in the query

In [11]:
data.obsm["protein_expression"].iloc[train_protein_expression.shape[0]:, ] = np.zeros_like(test_protein_expression)
data.obsm["protein_expression"]

,ADT_4-1BBLigand(CD137L)_A0890,ADT_CD102_A0104,ADT_CD103_A0201,ADT_CD105_A0812,ADT_CD106_A0226,ADT_CD107a(LAMP-1)_A0905,ADT_CD115(CSF-1R)_A0105,ADT_CD117(c-Kit)_A0012,ADT_CD11a_A0595,ADT_CD11b_A0014,...,ADT_TCRVr3_A0210,ADT_TCRbchain_A0120,ADT_TCRr-d_A0121,ADT_TER-119-ErythroidCells_A0122,ADT_TIGIT(Vstm3)_A0848,ADT_TLR4(CD284)-MD2Complex_A0875,ADT_Tim-4_A0567,ADT_XCR1_A0568,ADT_anti-P2RY12_A0415,ADT_integrinb7_A0214
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGGTATCTC-2,1.0,11.0,8.0,4.0,1.0,1.0,0.0,6.0,70.0,1.0,...,6.0,59.0,1.0,2.0,15.0,3.0,12.0,4.0,5.0,15.0
AAACCCACATTCTGTT-2,3.0,17.0,1.0,3.0,5.0,12.0,5.0,3.0,40.0,3.0,...,6.0,0.0,2.0,0.0,10.0,2.0,11.0,3.0,4.0,31.0
AAACCCAGTAGCTTGT-2,5.0,19.0,3.0,4.0,1.0,3.0,4.0,6.0,66.0,1.0,...,5.0,52.0,3.0,1.0,6.0,0.0,4.0,1.0,4.0,21.0
AAACCCAGTATCAGCT-2,1.0,33.0,6.0,4.0,3.0,0.0,4.0,5.0,36.0,2.0,...,10.0,4.0,0.0,1.0,8.0,5.0,5.0,5.0,2.0,32.0
AAACCCAGTTTCTATC-2,0.0,18.0,5.0,5.0,2.0,7.0,1.0,2.0,32.0,6.0,...,7.0,1.0,5.0,0.0,14.0,1.0,1.0,3.0,8.0,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGTTGGTGCTCTTC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGTTGTCAGCCTTC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Select highly variable genes

In [12]:
sc.pp.highly_variable_genes(data, batch_key="batch_index", flavor="seurat_v3", n_top_genes=4000, subset=True)

## Initialize model

In [13]:
scvi.model.TOTALVI.setup_anndata(data, batch_key="batch_index", protein_expression_obsm_key="protein_expression", protein_names_uns_key="protein_names")
model = scvi.model.TOTALVI(data, latent_distribution="normal", n_layers_decoder=2)

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


INFO     Using column names from adata.uns['protein_names']                                                        
INFO     Found batches with missing protein expression                                                             
INFO     Computing empirical prior initialization for protein background.                                          


/home/hongyj/.conda/envs/scvi_tools/lib/python3.8/site-packages/scvi/model/_totalvi.py:130: UserWarning: Some proteins have all 0 counts in some batches. These proteins will be treated as missing measurements; however, this can occur due to experimental design/biology. Reinitialize the model with `override_missing_proteins=True`,to override this behavior.
  warnings.warn(msg, UserWarning)


## Start training

In [14]:
model.train(lr=0.004)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 400/400: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400/400 [15:20<00:00,  1.73s/it, loss=602, v_num=1]

`Trainer.fit` stopped: `max_epochs=400` reached.


Epoch 400/400: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400/400 [15:20<00:00,  2.30s/it, loss=602, v_num=1]


## Perform prediction on the query

In [15]:
_, protein_means = model.get_normalized_expression(n_samples=25, transform_batch=0, include_protein_background=True, sample_protein_mixing=False, return_mean=True)
predicted_protein_expression = protein_means.loc[test_protein_expression.index, :]
predicted_protein_expression.to_csv(f"{base_dir}/test_predicted_protein_expression (TotalVI).csv")
model.save(dir_path=f"{base_dir}/TotalVI_model", save_anndata=True, overwrite=True)
predicted_protein_expression

,ADT_4-1BBLigand(CD137L)_A0890,ADT_CD102_A0104,ADT_CD103_A0201,ADT_CD105_A0812,ADT_CD106_A0226,ADT_CD107a(LAMP-1)_A0905,ADT_CD115(CSF-1R)_A0105,ADT_CD117(c-Kit)_A0012,ADT_CD11a_A0595,ADT_CD11b_A0014,...,ADT_TCRVr3_A0210,ADT_TCRbchain_A0120,ADT_TCRr-d_A0121,ADT_TER-119-ErythroidCells_A0122,ADT_TIGIT(Vstm3)_A0848,ADT_TLR4(CD284)-MD2Complex_A0875,ADT_Tim-4_A0567,ADT_XCR1_A0568,ADT_anti-P2RY12_A0415,ADT_integrinb7_A0214
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGCGATCGA-1,2.299853,16.978725,21.582970,2.776455,3.079024,2.392051,2.492142,5.205436,61.913509,2.693110,...,6.556395,22.066334,3.126842,0.722472,7.896860,2.669879,4.857885,2.681127,3.928514,51.167812
AAACCCACACCTCTAC-1,2.301101,25.803032,5.174891,2.667592,3.160701,2.450279,2.633633,3.237055,45.729321,3.182180,...,6.598494,3.032140,3.158726,0.776638,7.661255,2.549407,5.622162,2.686385,3.974387,14.344669
AAACCCACACGTCGTG-1,2.455137,42.108105,5.209652,2.871830,3.183123,2.529065,2.478962,3.170623,23.003263,2.732913,...,6.515411,3.074784,3.214071,0.756270,7.777983,2.646589,5.015903,2.642762,3.987291,15.169168
AAACCCACATGTCTAG-1,2.718896,51.029396,5.448532,2.831261,3.383618,3.574743,2.706374,3.326723,39.051216,3.246922,...,6.473286,3.376965,3.448881,1.139016,8.779364,2.651730,6.786123,3.107562,4.496990,13.799274
AAACCCAGTCAAAGTA-1,2.414376,31.367760,5.213583,2.666457,3.149922,2.512016,2.512097,3.171805,42.418037,3.029922,...,6.519316,3.103623,3.164415,0.755673,8.020616,2.678121,5.024592,2.707331,3.921428,20.488699
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,2.312876,26.378025,5.144692,2.668630,3.135773,2.450809,2.447504,3.245957,26.299711,2.708125,...,6.538975,3.029050,3.199696,0.755022,7.874843,2.550714,4.856208,2.688628,3.923270,16.571810
TTTGTTGGTGCTCTTC-1,2.327827,27.560196,5.242903,2.648187,3.072678,2.483044,2.516240,3.260113,31.373306,2.666587,...,6.435497,3.068546,3.163941,0.752894,7.963322,2.569937,4.828028,2.692855,4.010351,17.844578
TTTGTTGTCAGCCTTC-1,2.311840,17.282642,5.237436,2.640607,3.067701,2.388124,2.501643,4.090049,53.204044,2.704233,...,6.596019,31.244331,3.197385,0.736623,7.591192,2.553535,4.884826,2.675026,3.989270,12.996071
